# Simple baseline (CNN)

This notebook tries to use CNNs to improve our previous baseline of 64%.

In [1]:
%load_ext autoreload
%autoreload 2

# autoreload ensures our own code can be reloaded if we hit bugs.

In [14]:
  %reload_ext autoreload

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from chess_classifier.data.df_loader import load_df_from_parquet

In [8]:
# We've extracted out the data loading into its own function.

df = load_df_from_parquet("games.parquet", 100_000, shuffle_seed=0)

,lichess_id,evals,ply_count,white,black,white_rating,black_rating,time_initial,time_increment,result,...,g7,h7,a8,b8,c8,d8,e8,f8,g8,h8
0,eh5dKZuD,<NA>,29,Panchito0O,PauloPeru78,1247,1218,180,0,1-0,...,p,p,r,n,b,q,k,b,n,r
1,4mp4HhGa,<NA>,84,Guzman70,Issam_k,2352,2352,60,0,0-1,...,,p,,,,,,b,k,
2,hrtbQoGE,"[18, 25, 0, 121, 39, 98, 85, 95, 78, 110, 98, ...",18,begezavr,DannyE2E4,1703,1765,300,3,0-1,...,p,p,r,,b,,k,b,,r
3,IeES7auV,<NA>,121,Y0uNg_12345,cheetosss,2164,2374,15,0,1-0,...,k,p,,,,,,,,
4,ps2JBUV1,<NA>,42,Van_Eck,Baumberger,2447,2342,60,0,0-1,...,p,p,r,n,,q,k,b,,r
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,pYJig8Xo,<NA>,40,khottifa,ariadni1212,1684,1705,300,0,0-1,...,p,p,r,,b,,,r,k,
99996,jtMaoKN1,<NA>,68,KKorbeusz,ValeryGros,1444,1488,60,0,0-1,...,,p,,,,,,Q,,
99997,KFCeW57G,<NA>,63,Veqxh,olajiks,1359,1408,180,2,1-0,...,p,p,r,,,,,r,k,
99998,Rre38e3O,<NA>,113,dvoscar,Battousay01,2257,2181,60,0,1-0,...,k,,,,,,,,,


In [16]:
class ChessCNN(nn.Module):
    def __init__(self, tabular_features_len=4):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(12, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8 + tabular_features_len, 128),
            nn.ReLU(),
            nn.Linear(128, 3),
        )
    
    def forward(self, board, tab_features):
        x = self.conv(board)  # (batch, 12, 8, 8) -> (batch, 64, 8, 8)
        x = torch.cat([x.flatten(1), tab_features], dim=1)
        return self.fc(x)

In [34]:
result_map = {'1-0': 0, '0-1': 1, '1/2-1/2': 2}

def preprocess_df(df):
    # Select out the features and labels
    labels = 'result'
    
    categorical_features = [chr(file+ord('a'))+str(rank+1) for rank in range(8) for file in range(8)]
    features = ['white_rating', 'black_rating', 'ply']
    
    one_hot_df = pd.DataFrame({f'{col}_{piece}': df[col] == piece for piece in list('prnbqkPRNBQK') for col in categorical_features})
    X_df = pd.concat([df[features], one_hot_df], axis='columns')
    X_df['ply'] = X_df['ply'] / 80 # average game length
    X_df['move'] = (X_df['ply'] % 2 == 0) # turn (black or white)
    X_df['white_rating'] = (X_df['white_rating'] - 1500) / 400 
    X_df['black_rating'] = (X_df['black_rating'] - 1500) / 400

    y_df = df[labels].map(result_map)
    return X_df, y_df

# Note: this function has changed from the previous notebook
def preprocess(X_df, y_df, test_size=0.2, random_state=None):
    tabular_features = ['ply', 'move', 'white_rating', 'black_rating']

    tensor = X_df.drop(tabular_features, axis='columns').to_numpy().reshape(-1, 8,8,12)
    
    X, X_tab, y = np.transpose(tensor, axes=(0, -1, 1, 2)), X_df[tabular_features].to_numpy(), y_df.to_numpy()
    return train_test_split(X, X_tab, y, test_size=test_size, random_state=random_state)

def train_baseline(X_train, X_tab_train, y_train) -> ChessCNN:
    model = ChessCNN(tabular_features_len=X_tab_train.shape[-1])
    model.fit(X_train, y_train)
    return model

def eval_model(model, X_test, y_test):
    y_preds = model.predict(X_test)
    num_correct = (y_preds == y_test).sum()
    total = y_test.shape[0]
    print(f"Accuracy: {num_correct}/{total} = {(num_correct / total):.3f}")
    return dict(num_correct=num_correct, total=total, accuracy=num_correct / total)

In [35]:
X_df, y_df = preprocess_df(df)
X_train, X_test, X_tab_train, X_tab_test, y_train, y_test = preprocess(X_df, y_df) 

In [33]:
X_train.shape, X_tab_train.shape 

((80000, 12, 8, 8), (80000, 4))

In [ ]:
# shape seems correct. Now, let's set up a simple training loop:

board_tensor = torch.tensor(X_train).float()
tab_tensor = torch.tensor(X_tab_train).float()
y_tensor = torch.tensor(y).long()

dataset = TensorDataset(board_tensor, tab_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

model = ChessCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    model.train()
    for board, tab, labels in loader:
        optimizer.zero_grad()
        outputs = model(board, tab)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        outputs = model(X_test, X_tab_test)
        loss_test = criterion(outputs, labels)
        test_acc = (y_preds == y_test).sum() / y_test.shape[0]
        print(f"{loss_test=}, {test_acc=}")
        

In [12]:
X_df, y_df = preprocess_df(df)


X_train, X_test, y_train, y_test = preprocess(X_df, y_df, random_state=0)
model = train_baseline(X_train, y_train)
_ = eval_model(model, X_test, y_test)

Accuracy: 11636/20000 = 0.582


### Accuracy jump, weight visualization

We've improved accuracy again just by adding piece positions (61% -> 64%). So, this means that our model is benefitting from being aware of non-linear interactions between pieces. Let's try adding some features and tuning and see if we can't squeeze out a couple of more points.

In [30]:
def train_baseline(X_train, y_train, **kwargs):
    model = XGBClassifier(**kwargs)
    model.fit(X_train, y_train)
    return model

X_df, y_df = preprocess_df(df)
X_train, X_test, y_train, y_test = preprocess(X_df, y_df, random_state=0)
model = train_baseline(X_train, y_train, n_estimators=500, max_depth=8, learning_rate=0.1)
_ = eval_model(model, X_test, y_test)

Accuracy: 12939/20000 = 0.647


In [31]:
# Minor improvement. Let's see about overfitting.
_ = eval_model(model, X_train, y_train)

Accuracy: 61827/80000 = 0.773


In [34]:
# Clearly, we are starting to overfit. More data will likely help.
df.shape[0]

100000

In [44]:
# Let's scale up the data 10x.

games = duckdb.read_parquet("games.parquet")
subset = duckdb.sql("FROM games WHERE result != '*' LIMIT 1000000")
game_positions = duckdb.sql("""
SELECT
  lc.* EXCLUDE (movedata, clocks_white, clocks_black, tournament),
  t.ply,
  UNNEST(board_at_position(lc.movedata, t.ply))
FROM subset AS lc
CROSS JOIN LATERAL (
  SELECT 1 + CAST(floor(random() * lc.ply_count) AS INTEGER) AS ply
) AS t ORDER BY random()
""")
df = game_positions.df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [45]:
X_df, y_df = preprocess_df(df)
X_train, X_test, y_train, y_test = preprocess(X_df, y_df, random_state=0)
model = train_baseline(X_train, y_train, n_estimators=500, max_depth=8, learning_rate=0.1)
_ = eval_model(model, X_test, y_test)

Accuracy: 128524/200000 = 0.643


In [46]:
_ = eval_model(model, X_train, y_train)

Accuracy: 543352/800000 = 0.679


In [47]:
# Let's try scaling up even more.
X_df, y_df = preprocess_df(df)
X_train, X_test, y_train, y_test = preprocess(X_df, y_df, random_state=0)
model = train_baseline(X_train, y_train, n_estimators=1000, max_depth=8, learning_rate=0.1)
_ = eval_model(model, X_test, y_test)

Accuracy: 129492/200000 = 0.647


In [48]:
_ = eval_model(model, X_train, y_train)

Accuracy: 568614/800000 = 0.711


In [49]:
# Let's try scaling up even more.
# X_df, y_df = preprocess_df(df)
# X_train, X_test, y_train, y_test = preprocess(X_df, y_df, random_state=0)
model = train_baseline(X_train, y_train, n_estimators=1000, max_depth=10, learning_rate=0.1)
_ = eval_model(model, X_test, y_test)

Accuracy: 130116/200000 = 0.651


### Conclusion

We've improved accuracy again by capturing non-linearities via XGBoost. This has resulted in a test accuracy improvement of between 3-4%. The only problem
is that we are still overfitting -- even after scaling up to a million positions.

A few directions we could try:
1. Scaling up data and the model further. Given that we barely got an improvement of 1% after scaling data 10x, it's likely we are beginning to hit diminishing
returns.
2. Add time-control / clock features: given blunder rates in fast time controls, it's likely that this would improve model prediction a non-trivial amount.
3. Try a different model architecture: Chess is fundamentally a spatial game -- it's possible that a deep neural network such as a CNN on a spatial board could
improve performance.

The next notebook will explore 